# Aula 1 - Problema e primeiro pipeline
## Do dado bruto a primeira decisao computacional

**Objetivo:** Configurar o ambiente e transformar uma imagem da camera em uma primeira decisao computacional.

**Atividade do dia:** Capturar video da camera, detectar uma cor ou marcador visual e produzir uma classificacao generica de posicao. A aplicacao ao projeto final fica para depois da escolha do grupo.

---


## Bibliotecas que vamos usar

Antes de programar, vamos entender quais bibliotecas Python usaremos neste projeto e para que cada uma serve.

### Bibliotecas de hoje

| Biblioteca | Para que serve |
|-----------|---------------|
| **NumPy** | Manipular arrays (matrizes) de numeros. Uma imagem digital nada mais e que uma matriz NumPy. Cada pixel e um numero. |
| **OpenCV** (`cv2`) | Biblioteca de visao computacional. Abre a camera, le imagens, converte espacos de cor (BGR, HSV, cinza), detecta contornos e desenha na tela. |
| **matplotlib** (`plt`) | Gera graficos e mostra imagens dentro do notebook. |

### Bibliotecas das proximas aulas (ja instalamos hoje)

| Biblioteca | Para que serve | Aula |
|-----------|---------------|------|
| **pandas** | Manipular dados em tabela (DataFrame). Ler e escrever CSV com metadados e resultados. | 2, 5, 8 |
| **scikit-learn** | Aprendizado de maquina classico. Treina Decision Tree, Random Forest, SVM e k-NN. Calcula acuracia, precisao, revocacao. | 3, 5 |
| **seaborn** | Graficos estatisticos. Usaremos para plotar a matriz de confusao de forma colorida. | 3, 5 |
| **joblib** | Salvar e carregar modelos treinados. Permite reusar o modelo na inferencia sem retreinar. | 3, 6 |
| **PyTorch** (`torch`) | Framework de deep learning. Define e treina redes neurais. Suporta GPU para acelerar o treino. | 4 |
| **torchvision** | Modelos de visao pre-treinados (MobileNetV2, ResNet) e ferramentas de imagem para PyTorch. | 4 |
| **pyserial** (`serial`) | Comunicacao serial pela porta USB com Arduino ou ESP32. Sera usada quando o projeto escolhido precisar de hardware. | 6, 7 |

### Analogia

Pense no projeto como uma cozinha:

- **NumPy** = ingredientes basicos (tudo comeca aqui)
- **OpenCV** = faca e tabua (processa as imagens)
- **scikit-learn** = forno simples (treina o baseline)
- **PyTorch** = forno profissional (treina redes neurais)
- **matplotlib/seaborn** = os pratos (apresenta o resultado)
- **pyserial** = o entregador (leva o comando ao Arduino)
- **pandas** = caderno de receitas (organiza os dados)


In [ ]:
# Importar todas as bibliotecas do projeto e verificar versoes

# --- Bibliotecas de hoje ---
import numpy as np
print(f'NumPy {np.__version__} - arrays e matrizes numericas')

import cv2
print(f'OpenCV {cv2.__version__} - visao computacional e camera')

import matplotlib
print(f'matplotlib {matplotlib.__version__} - graficos e visualizacao')

import matplotlib.pyplot as plt

# --- Bibliotecas das proximas aulas ---
import pandas as pd
print(f'pandas {pd.__version__} - manipulacao de dados em tabela')

import sklearn
print(f'scikit-learn {sklearn.__version__} - aprendizado de maquina classico')

import seaborn as sns
print(f'seaborn {sns.__version__} - graficos estatisticos')

import joblib
print(f'joblib {joblib.__version__} - salvar/carregar modelos')

import torch
print(f'PyTorch {torch.__version__} - deep learning e redes neurais')
if torch.cuda.is_available():
    print(f'  GPU disponivel: {torch.cuda.get_device_name(0)}')
else:
    print(f'  GPU nao disponivel - usaremos CPU')

import torchvision
print(f'torchvision {torchvision.__version__} - modelos de visao pre-treinados')

import serial
print(f'pyserial {serial.__version__} - comunicacao com Arduino')

print()
print('Todas as bibliotecas carregadas com sucesso!')


## Passo 1: Verificar o ambiente

In [ ]:
import sys
print(f"Python: {sys.version}")

import numpy as np
import cv2
print(f"NumPy: {np.__version__}")
print(f"OpenCV: {cv2.__version__}")
print("Ambiente OK!")


## Passo 2: Testar a camera

In [ ]:
# Abrir a camera e mostrar video
# Pressione Q para sair

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("ERRO: Nao foi possivel abrir a camera.")
else:
    print("Camera aberta! Pressione Q na janela para sair.")
    while True:
        ret, frame = cap.read()
        if not ret:
            print("ERRO: Nao foi possivel ler o quadro.")
            break
        
        cv2.imshow('Camera', frame)
        
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()
print("Camera fechada.")


## Passo 3: Converter para HSV

In [ ]:
# Entender os espacos de cor

cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    cinza = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    
    # Mostrar os 3 canais HSV separados
    h, s, v = cv2.split(hsv)
    
    cv2.imshow('Original', frame)
    cv2.imshow('HSV', hsv)
    cv2.imshow('Cinza', cinza)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## Passo 4: Detectar uma cor (verde)

In [ ]:
# Detectar a cor verde usando HSV

import numpy as np

cap = cv2.VideoCapture(0)

# Intervalo de verde no espaco HSV
verde_baixo = np.array([40, 50, 50])
verde_alto = np.array([80, 255, 255])

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    
    # Criar mascara
    mascara = cv2.inRange(hsv, verde_baixo, verde_alto)
    
    # Aplicar mascara no frame original
    resultado = cv2.bitwise_and(frame, frame, mask=mascara)
    
    # Encontrar contornos
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if contornos:
        maior = max(contornos, key=cv2.contourArea)
        area = cv2.contourArea(maior)
        
        if area > 500:
            # Calcular centro
            M = cv2.moments(maior)
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            
            # Desenhar
            cv2.circle(frame, (cx, cy), 20, (0, 255, 0), 3)
            cv2.drawContours(frame, [maior], -1, (0, 255, 0), 2)
            cv2.putText(frame, f"Area: {area:.0f}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    
    cv2.imshow('Deteccao', frame)
    cv2.imshow('Mascara', mascara)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()


## Passo 5: Produzir uma classificacao generica

In [ ]:
# Classificar a posicao do objeto: ESQUERDA, CENTRO ou DIREITA

import numpy as np

cap = cv2.VideoCapture(0)

verde_baixo = np.array([40, 50, 50])
verde_alto = np.array([80, 255, 255])

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    h, w = frame.shape[:2]
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    mascara = cv2.inRange(hsv, verde_baixo, verde_alto)
    
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    comando = "SEM_DETECCAO"
    cx, cy = None, None
    
    if contornos:
        maior = max(contornos, key=cv2.contourArea)
        area = cv2.contourArea(maior)
        
        if area > 500:
            M = cv2.moments(maior)
            if M["m00"] > 0:
                cx = int(M["m10"] / M["m00"])
                cy = int(M["m01"] / M["m00"])
                
                centro = w // 2
                margem = 80
                
                if cx < centro - margem:
                    comando = "ESQUERDA"
                elif cx > centro + margem:
                    comando = "DIREITA"
                else:
                    comando = "CENTRO"
    
    # Desenhar resultado
    cores = {
        "ESQUERDA": (0, 200, 255),
        "DIREITA": (255, 100, 0),
        "CENTRO": (0, 255, 0),
        "SEM_DETECCAO": (0, 0, 255),
    }
    
    if cx is not None:
        cv2.circle(frame, (cx, cy), 20, cores[comando], 3)
    
    cv2.putText(frame, comando, (10, 60),
                cv2.FONT_HERSHEY_SIMPLEX, 1.5, cores[comando], 3)
    
    cv2.imshow('Classificacao visual - Aula 1', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("Experimento finalizado!")


## Entrega da Aula 1

- [ ] Notebook executado sem erros
- [ ] Camera abre e detecta a cor
- [ ] Classificacao (ESQUERDA/CENTRO/DIREITA) funciona
- [ ] Salvar uma captura de tela
- [ ] Atualizar o README.md com o problema escolhido pelo grupo

**Extensao:** Diagnosticar a demanda e definir o publico (3 h previstas).
